***Logistic Regression***

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

def prepare_data_logistic():
    df = pd.read_csv('Part_2_DataCollection/Chuot_PhongVu.csv')

    # Trích xuất đặc trưng từ tên chuột
    df['Is_Gaming'] = df['Name'].str.contains('gaming', case=False, regex=True).astype(np.float32)
    df['Is_Wireless'] = df['Name'].str.contains(r'không dây|wireless|bluetooth|2\.4g', case=False, regex=True).astype(np.float32)
    df['Is_Silent'] = df['Name'].str.contains('silent|chống ồn|yên tĩnh', case=False, regex=True).astype(np.float32)

    # Giữ lại top 10 hãng phổ biến nhất, các hãng còn lại thì là 'Other'
    top_brands = df['Brand'].value_counts().nlargest(10).index
    df['Brand_Grouped'] = df['Brand'].apply(lambda x: x if x in top_brands else 'Other')

    # One-Hot Encoding
    brand_dummies = pd.get_dummies(df['Brand_Grouped'], prefix='Brand', drop_first=True).astype(np.float32)

    # Dùng Scaler chuẩn hóa
    scaler = StandardScaler()
    retail_price_scaled = scaler.fit_transform(df[['Retail_Price']]).astype(np.float32)
    df_price_scaled = pd.DataFrame(retail_price_scaled, columns=['Retail_Price_Scaled'], index=df.index)

    # Kết nối lại tất cả đặc trưng và đưa vào ma trận X
    X_df = pd.concat([df_price_scaled, df[['Is_Gaming', 'Is_Wireless', 'Is_Silent']], brand_dummies], axis=1)
    return df, X_df, scaler

df, X_df, scaler = prepare_data_logistic()

y = df['Is_Gaming'].values.astype(np.float32).ravel()
X = X_df.drop(columns=['Is_Gaming']).values.astype(np.float32)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = {
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-score': f1_score(y_test, y_pred, zero_division=0),
}

print('Kết quả đánh giá: ')
for metric_name, metric_value in metrics.items():
    print(f'{metric_name}: {metric_value}')

Kết quả đánh giá: 
Precision: 0.803921568627451
Recall: 0.803921568627451
F1-score: 0.803921568627451


***Softmax Regression***

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

def prepare_data_softmax():
    df = pd.read_csv('Part_2_DataCollection/Chuot_PhongVu.csv')

    # Trích xuất đặc trưng từ tên chuột
    df['Is_Gaming'] = df['Name'].str.contains('gaming', case=False, regex=True).astype(np.float32)
    df['Is_Wireless'] = df['Name'].str.contains(r'không dây|wireless|bluetooth|2\.4g', case=False, regex=True).astype(np.float32)
    df['Is_Silent'] = df['Name'].str.contains('silent|chống ồn|yên tĩnh', case=False, regex=True).astype(np.float32)
    df['Is_RGB'] = df['Name'].str.contains(r'rgb|led', case=False, regex=True).astype(np.float32)
    df['DPI_Mentioned'] = df['Name'].str.contains(r'dpi|\d{4,5}', case=False, regex=True).astype(np.float32)

    # Xử lý cột Discount_Percent
    df['Discount_Ratio'] = (df['Discount_Percent']/100.0).astype(np.float32)

    # Scale số tiền giảm giá
    scaler = StandardScaler()
    discount_amount_scaler = scaler.fit_transform(df[['Discount_Amount']]).astype(np.float32)
    df_discount_scaled = pd.DataFrame(discount_amount_scaler, columns=['Discount_Amount_Scaled'], index=df.index)

    # Mã hóa One-Hot Encoding cho top 5 hãng
    top_brands = df['Brand'].value_counts().nlargest(5).index
    df['Brand_Grouped'] = df['Brand'].apply(lambda x: x if x in top_brands else 'Other')
    brand_dummies = pd.get_dummies(df['Brand_Grouped'], prefix='Brand', drop_first=True).astype(np.float32)

    # Gom tất cả vào ma trận X
    X_df = pd.concat([df[['Is_Gaming', 'Is_Wireless', 'Is_Silent', 'Is_RGB', 'Discount_Ratio', 'DPI_Mentioned']], df_discount_scaled, brand_dummies], axis=1)

    # Phân khúc giá thành 5 lớp: 0, 1, 2, 3, 4
    bins = [-1, 250_000, 500_000, 1_000_000, 2_000_000, np.inf]
    y = pd.cut(df['Retail_Price'], bins=bins, labels=[0, 1, 2, 3, 4]).astype(int).values

    return df, X_df, y

df, X_df, y= prepare_data_softmax()
X_train, X_test, y_train, y_test = train_test_split(X_df.values.astype(np.float32), y.ravel(), test_size=0.2, random_state=42, stratify=y)

model = LogisticRegression(C=np.inf, solver='lbfgs', max_iter=1000, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = {
    'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
    'Recall': recall_score(y_test, y_pred, average='weighted',zero_division=0),
    'F1-score': f1_score(y_test, y_pred, average='weighted',zero_division=0),
}

print('Kết quả đánh giá: ')
for metric_name, metric_value in metrics.items():
    print(f'{metric_name}: {metric_value}')

Kết quả đánh giá: 
Precision: 0.9215758241758242
Recall: 0.92
F1-score: 0.919492303992304
